<a href="https://colab.research.google.com/github/Viditp25/flyrank-intern-/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

**Purpose:** audit the Week-5 model’s validation, feature leakage, failure cases, and claim strength. The notebook compares a stratified random-row split with a client-grouped split using the same Logistic Regression model and features.

Dataset: FlyRank anonymized starter CSV (`content_refresh_anonymized.csv`). No client names, URLs, titles, or private queries are displayed.

Research source: [FlyRank — The State of AI-Driven SEO, March 2026](https://github.com/flyrank-bih/flyrank-ml-internship-starter/blob/main/docs/flyrank-seo-research-march-2026.pdf).


## 1. Two paper findings + my methodology questions

### Finding A — Finding #1, “The Anatomy of Growing Content” (paper page 6)

The paper reports that the rising-impressions cohort averaged about **3,180 words** and **184 days old**, compared with about **2,311 words** and **230 days old** for the declining cohort. It reports 74,187 rising pages and 45,272 declining pages and explicitly frames the result as observational.

**Constructive methodology questions**
1. **Where does the label come from?** Trend direction is defined from recent 30-day impressions compared with the previous 30 days. Are word count and age timestamped before that outcome window, or are they snapshot fields with uncertain timing?
2. **Does the design support the claim?** This large cohort comparison supports describing differences in this portfolio. It does not establish that adding words or changing a page’s age causes it to grow.
3. **What would strengthen it?** Document feature timestamps, report distributions and uncertainty alongside means, and test whether earlier features predict a later trend in a time-forward holdout.

### Finding B — Finding #4, “The Freshness Multiplier” (paper page 9)

The paper reports a **7.88:1** growth-to-decline ratio for the 31–90-day freshness window. It also reports **283:1** in the 361+ day bucket, while noting that it contains 283 growing pages and only **one declining page**. For 365+ day content refreshed within 30 days, it reports 3.2× health and 57× impressions.

**Constructive methodology questions**
1. **Where does the label come from?** Growth/decline is based on recent impression change, while freshness is days since the last update. Can we confirm the update timestamp precedes the performance window used to define the outcome?
2. **Does the design support the claim?** The 361+ ratio is unstable because only one page is declining. For the 3.2× and 57× comparisons, were refreshed pages compared with similar unrefreshed pages matched on prior traffic, depth, age, and intent? Otherwise, stronger pages may be more likely to be refreshed; the association does not prove refresh caused the difference.
3. **What would strengthen it?** Show counts beside ratios, set minimum sample sizes, and compare similar pages with a pre/post design and a suitable control group.

These are constructive questions about evidence and interpretation, not a grade of the paper.


In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix

RANDOM_STATE = 42
DATA_REL = Path("data/raw/content_refresh_anonymized.csv")
DATA_URL = "https://raw.githubusercontent.com/Viditp25/flyrank-intern-/main/data/raw/content_refresh_anonymized.csv"
def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / DATA_REL).is_file(): return candidate
    return None
repo_root = find_repo_root()
data_source = str(repo_root / DATA_REL) if repo_root else DATA_URL
df = pd.read_csv(data_source)
required = ["client_id", "content_id", "trend_direction", "impressions_90d", "impressions_last_30d", "impressions_prev_30d", "clicks_90d", "clicks_last_30d", "clicks_prev_30d", "sessions_90d", "sessions_last_30d", "sessions_prev_30d"]
missing = sorted(set(required) - set(df.columns))
if missing: raise ValueError(f"Required dataset columns missing: {missing}")
print(f"Rows: {len(df):,} | Columns: {len(df.columns):,}")
labels = df["trend_direction"].astype("string").str.strip().str.lower()
valid_labels = {"down", "stable", "up", "new", "flat"}
if (labels.isna() | ~labels.isin(valid_labels)).any(): raise ValueError("Missing or unexpected trend_direction labels.")
df["target"] = labels.eq("down").astype(int)
display(df["target"].value_counts().sort_index().rename_axis("target").to_frame("rows"))
print(f"Positive/base rate (target = declining): {df.target.mean():.2%}")


Rows: 30,000 | Columns: 44


,rows
target,
0,13738
1,16262


Positive/base rate (target = declining): 54.21%


## 2. My model under an honest split (before/after)

**Before:** stratified random-row split. Pages from the same client can occur in both train and test, so the evaluation may benefit from client-specific patterns.

**After:** `GroupShuffleSplit` by `client_id`. The test set contains clients unseen during training. Both use the same Logistic Regression pipeline, feature definitions, and metrics. Since the test populations differ, the metric gap is a diagnostic rather than a perfectly controlled estimate of memorization.


In [14]:
# Reconstruct the 61–90 day historical window, following Week 5.
for metric in ["impressions", "clicks", "sessions"]:
    total = pd.to_numeric(df[f"{metric}_90d"], errors="coerce")
    last = pd.to_numeric(df[f"{metric}_last_30d"], errors="coerce")
    prev = pd.to_numeric(df[f"{metric}_prev_30d"], errors="coerce")
    older = total - last - prev
    invalid = total.isna() | last.isna() | prev.isna() | older.lt(0)
    df[f"{metric}_days_61_90"] = older.mask(invalid)
model_df = df.loc[df.client_id.notna() & df.content_id.notna()].copy()
for metric in ["impressions", "clicks", "sessions"]:
    model_df[f"{metric}_pre"] = pd.to_numeric(model_df[f"{metric}_prev_30d"], errors="coerce") + pd.to_numeric(model_df[f"{metric}_days_61_90"], errors="coerce")
impr_pre = pd.to_numeric(model_df.impressions_pre, errors="coerce")
click_pre = pd.to_numeric(model_df.clicks_pre, errors="coerce")
model_df["ctr_pre"] = click_pre / impr_pre.replace(0, np.nan)
older_impr = pd.to_numeric(model_df.impressions_days_61_90, errors="coerce")
prev_impr = pd.to_numeric(model_df.impressions_prev_30d, errors="coerce")
model_df["prior_momentum_pct"] = ((prev_impr - older_impr) / older_impr.replace(0, np.nan)) * 100
log_sources = ["impressions_days_61_90", "clicks_days_61_90", "sessions_days_61_90", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "impressions_pre", "clicks_pre", "sessions_pre"]
for col in log_sources:
    vals = pd.to_numeric(model_df[col], errors="coerce")
    model_df[f"log1p_{col}"] = np.log1p(vals.where(vals >= 0))
numeric_candidates = ["search_volume", "competition", "cpc", "word_count", "char_count", "impressions_days_61_90", "clicks_days_61_90", "sessions_days_61_90", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "impressions_pre", "clicks_pre", "sessions_pre", "ctr_pre", "prior_momentum_pct"] + [f"log1p_{c}" for c in log_sources]
categorical_candidates = ["competition_level", "content_type", "main_intent", "word_count_tier"]
forbidden = {"client_id", "content_id", "trend_direction", "target", "trend_pct", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "impressions_90d", "clicks_90d", "sessions_90d"}
numeric_features = [c for c in numeric_candidates if c in model_df.columns and c not in forbidden]
categorical_features = [c for c in categorical_candidates if c in model_df.columns and c not in forbidden]
for c in numeric_features: model_df[c] = pd.to_numeric(model_df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
numeric_features = [c for c in numeric_features if model_df[c].notna().any()]
feature_cols = numeric_features + categorical_features
if not feature_cols: raise ValueError("No permitted features found.")
X = model_df[feature_cols].copy(); y = model_df.target.astype(int).copy(); groups = model_df.client_id.copy()
print(f"Rows used: {len(model_df):,} | Features: {len(feature_cols)} | Base rate: {y.mean():.2%}")
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


Rows used: 30,000 | Features: 29 | Base rate: 54.21%
Numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_days_61_90', 'clicks_days_61_90', 'sessions_days_61_90', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'impressions_pre', 'clicks_pre', 'sessions_pre', 'ctr_pre', 'prior_momentum_pct', 'log1p_impressions_days_61_90', 'log1p_clicks_days_61_90', 'log1p_sessions_days_61_90', 'log1p_impressions_prev_30d', 'log1p_clicks_prev_30d', 'log1p_sessions_prev_30d', 'log1p_impressions_pre', 'log1p_clicks_pre', 'log1p_sessions_pre']
Categorical features: ['competition_level', 'content_type', 'main_intent', 'word_count_tier']


In [15]:
def make_pipeline():
    transformers = []
    if numeric_features:
        transformers.append(("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features))
    if categorical_features:
        transformers.append(("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features))
    pre = ColumnTransformer(transformers=transformers, remainder="drop")
    return Pipeline([("preprocessor", pre), ("model", LogisticRegression(C=0.1, max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))])
def p_at_k(y_true, score, k):
    y_arr = np.asarray(y_true, dtype=int); score = np.asarray(score, dtype=float)
    if len(y_arr) < k: return np.nan
    return float(y_arr[np.argsort(-score, kind="stable")[:k]].mean())
def fit_evaluate(name, tr, te):
    pipe = make_pipeline(); pipe.fit(X.iloc[tr], y.iloc[tr]); prob = pipe.predict_proba(X.iloc[te])[:, 1]
    train_clients = set(model_df.iloc[tr].client_id); test_clients = set(model_df.iloc[te].client_id)
    result = {"Split": name, "Train rows": len(tr), "Test rows": len(te), "Train clients": len(train_clients), "Test clients": len(test_clients), "Client overlap": len(train_clients & test_clients), "Test base rate": float(y.iloc[te].mean()), "Precision@10": p_at_k(y.iloc[te], prob, 10), "Precision@50": p_at_k(y.iloc[te], prob, 50), "ROC-AUC": roc_auc_score(y.iloc[te], prob), "Average Precision": average_precision_score(y.iloc[te], prob), "Brier Score": brier_score_loss(y.iloc[te], prob)}
    return result, pipe, np.asarray(te), np.asarray(y.iloc[te]), np.asarray(prob)
all_idx = np.arange(len(model_df))
random_train, random_test = train_test_split(all_idx, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
random_result, random_pipe, random_test_idx, random_y, random_prob = fit_evaluate("Random row split (before)", random_train, random_test)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
group_train, group_test = next(splitter.split(X, y, groups=groups))
group_result, group_pipe, group_test_idx, group_y, group_prob = fit_evaluate("Client-grouped split (after)", group_train, group_test)
comparison = pd.DataFrame([random_result, group_result]).set_index("Split")
display(comparison.round(4))
assert random_result["Client overlap"] > 0, "Expected client overlap in random split; inspect dataset if zero."
assert group_result["Client overlap"] == 0, "Grouped split must have zero client overlap."
print("PASS: grouped split has zero client overlap.")


,Train rows,Test rows,Train clients,Test clients,Client overlap,Test base rate,Precision@10,Precision@50,ROC-AUC,Average Precision,Brier Score
Split,,,,,,,,,,,
Random row split (before),24000,6000,32,31,31,0.542,0.9,0.82,0.7109,0.7222,0.2142
Client-grouped split (after),23837,6163,25,7,0,0.511,0.8,0.66,0.6171,0.5988,0.2505


PASS: grouped split has zero client overlap.


### How to interpret the results

Read the **actual table above**, not assumed or copied numbers. A higher random-split score is consistent with the random split being an easier test when clients overlap; if the grouped score is similar or higher, report that observed result rather than forcing a memorization explanation.

- **Average Precision:** ranking quality for the declining class across precision/recall trade-offs.
- **ROC-AUC:** discrimination between positive and negative labels.
- **Precision@10 / @50:** observed share of declining pages among the top 10 / 50 ranked pages.
- **Brier Score:** probability error; lower is better.
- Compare metrics with the test base rate shown alongside them.

A client-grouped split tests transfer to unseen clients, but it is **not time-aware** and cannot prove all features were available before the outcome period.


## 3. Leakage audit

The target is `trend_direction == "down"`. The paper and repository guidance define trend direction from the recent 30-day impression change versus the previous 30 days, and `trend_pct` is label-derived. Therefore `trend_direction`, `trend_pct`, the target itself, IDs, current 30-day metrics, and 90-day totals are excluded from the final feature set.

The reconstructed 61–90-day and previous 30-day windows are intended as historical features. However, this notebook cannot independently verify collection timestamps or operational availability for every metadata field. `search_volume`, `competition`, `cpc`, word count, content type, and intent/tier fields must be checked against the prediction-time data contract before deployment claims are made.


In [16]:
audit = pd.DataFrame([{"Field": c, "In final feature set?": c in feature_cols, "Reason": "identifier / target / label-derived" if c in {"client_id", "content_id", "trend_direction", "target", "trend_pct"} else "overlaps or includes outcome window"} for c in sorted(forbidden)])
display(audit)
assert not audit["In final feature set?"].any()
assert not ({"trend_pct", "trend_direction", "target", "client_id", "content_id", "impressions_last_30d", "impressions_90d"} & set(feature_cols))
print("PASS: prohibited fields are excluded from final features.")

# Deliberately leaky diagnostic only: add trend_pct and observe its effect on the same grouped test split.
# This experiment is invalid for model selection and is never used as the final model.
if "trend_pct" in model_df.columns:
    leak_num = numeric_features + ["trend_pct"]
    leak_pre = ColumnTransformer(transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), leak_num),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ], remainder="drop")
    leak_pipe = Pipeline([("preprocessor", leak_pre), ("model", LogisticRegression(C=0.1, max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))])
    X_leak = model_df[leak_num + categorical_features].copy()
    X_leak[leak_num] = X_leak[leak_num].apply(pd.to_numeric, errors="coerce")
    leak_pipe.fit(X_leak.iloc[group_train], y.iloc[group_train])
    leak_prob = leak_pipe.predict_proba(X_leak.iloc[group_test])[:, 1]
    display(pd.DataFrame([
        {"Experiment": "Final feature set (no trend_pct)", "ROC-AUC": roc_auc_score(group_y, group_prob), "Average Precision": average_precision_score(group_y, group_prob)},
        {"Experiment": "DELIBERATELY LEAKY (+ trend_pct)", "ROC-AUC": roc_auc_score(group_y, leak_prob), "Average Precision": average_precision_score(group_y, leak_prob)},
    ]).set_index("Experiment").round(4))
    print("The second row is intentionally invalid. It demonstrates why label-derived fields must be excluded.")
else:
    print("trend_pct is absent from this dataset version; field-exclusion checks still apply.")


,Field,In final feature set?,Reason
0,clicks_90d,False,overlaps or includes outcome window
1,clicks_last_30d,False,overlaps or includes outcome window
2,client_id,False,identifier / target / label-derived
3,content_id,False,identifier / target / label-derived
4,impressions_90d,False,overlaps or includes outcome window
5,impressions_last_30d,False,overlaps or includes outcome window
6,sessions_90d,False,overlaps or includes outcome window
7,sessions_last_30d,False,overlaps or includes outcome window
8,target,False,identifier / target / label-derived
9,trend_direction,False,identifier / target / label-derived


PASS: prohibited fields are excluded from final features.


,ROC-AUC,Average Precision
Experiment,,
Final feature set (no trend_pct),0.6171,0.5988
DELIBERATELY LEAKY (+ trend_pct),0.9712,0.9735


The second row is intentionally invalid. It demonstrates why label-derived fields must be excluded.


## 4. Failure examples and claim rewrite

The next cell inspects false positives and false negatives from the **client-grouped** model at a 0.5 diagnostic threshold. This threshold is for error inspection only, not an asserted optimal operational threshold. Examples omit client and content identifiers.


In [17]:
errors = pd.DataFrame({"example": [f"test_example_{i+1:03d}" for i in range(len(group_test_idx))], "actual_decline": group_y.astype(int), "predicted_probability": group_prob.astype(float)})
errors["predicted_decline_at_0.5"] = (errors.predicted_probability >= 0.5).astype(int)
errors["error_type"] = np.select([(errors.actual_decline == 1) & (errors["predicted_decline_at_0.5"] == 0), (errors.actual_decline == 0) & (errors["predicted_decline_at_0.5"] == 1)], ["False negative", "False positive"], default="Correct")
test_features = model_df.iloc[group_test_idx].reset_index(drop=True)
for c in ["main_intent", "content_type", "word_count", "prior_momentum_pct"]:
    if c in test_features.columns: errors[c] = test_features[c].to_numpy()
print("Grouped-holdout error counts at threshold 0.5:")
display(errors.error_type.value_counts().rename_axis("error_type").to_frame("rows"))
print("False positives: predicted decline, observed label not down")
display(errors[errors.error_type == "False positive"].sort_values("predicted_probability", ascending=False).head(5).round(3))
print("False negatives: observed decline, model probability below threshold")
display(errors[errors.error_type == "False negative"].sort_values("predicted_probability", ascending=True).head(5).round(3))
tn, fp, fn, tp = confusion_matrix(errors.actual_decline, errors["predicted_decline_at_0.5"], labels=[0, 1]).ravel()
print(f"Confusion counts: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print("These are disagreements with a proxy label; they do not explain why a page changed.")


Grouped-holdout error counts at threshold 0.5:


,rows
error_type,
Correct,3500
False positive,1512
False negative,1151


False positives: predicted decline, observed label not down


,example,actual_decline,predicted_probability,predicted_decline_at_0.5,error_type,main_intent,content_type,word_count,prior_momentum_pct
3453,test_example_3454,0,0.991,1,False positive,informational,keyword article,2363.0,-69.123
5421,test_example_5422,0,0.973,1,False positive,informational,keyword article,NaN,-91.882
1513,test_example_1514,0,0.955,1,False positive,informational,keyword article,2784.0,-53.714
2799,test_example_2800,0,0.951,1,False positive,informational,keyword article,3472.0,NaN
2095,test_example_2096,0,0.949,1,False positive,informational,keyword article,1329.0,68.658


False negatives: observed decline, model probability below threshold


,example,actual_decline,predicted_probability,predicted_decline_at_0.5,error_type,main_intent,content_type,word_count,prior_momentum_pct
4504,test_example_4505,1,0.000,0,False negative,informational,keyword article,NaN,579.544
2413,test_example_2414,1,0.023,0,False negative,informational,keyword article,2890.0,186.110
1403,test_example_1404,1,0.023,0,False negative,informational,keyword article,NaN,22.818
5124,test_example_5125,1,0.070,0,False negative,informational,keyword article,NaN,-93.750
5005,test_example_5006,1,0.094,0,False negative,informational,keyword article,NaN,-88.462


Confusion counts: TN=1502, FP=1512, FN=1151, TP=1998
These are disagreements with a proxy label; they do not explain why a page changed.


### Claim rewrite

**Overstrong version:** “The model identifies which pages will decline and proves which pages should be refreshed.”

**Evidence-aligned version:** “In this dataset, the Logistic Regression model measured how well selected historical metrics and content metadata ranked pages labeled as declining under the dataset’s trend definition. Performance was evaluated on a client-grouped holdout and compared with a random-row split. The results are directional decision-support evidence for prioritizing manual review, not proof of future decline, causation, or the benefit of refreshing a particular page. Predictor timestamps still require confirmation, and a time-forward evaluation would be needed before making deployment claims.”

This wording is safer because the label is a proxy based on recent impression change, grouped validation tests unseen clients rather than future time, and observational associations do not prove causality.


In [18]:
# Final automated checks for the audit.
assert "trend_direction" not in feature_cols and "trend_pct" not in feature_cols and "target" not in feature_cols
assert "client_id" not in feature_cols and "content_id" not in feature_cols
assert group_result["Client overlap"] == 0
assert len(group_y) == len(group_prob) == len(group_test_idx)
assert np.isfinite(group_prob).all() and ((group_prob >= 0) & (group_prob <= 1)).all()
assert np.isfinite(comparison[["ROC-AUC", "Average Precision", "Brier Score"]].to_numpy(dtype=float)).all()
assert errors.error_type.isin(["Correct", "False positive", "False negative"]).all()
assert (tn + fp + fn + tp) == len(group_y)
print("=" * 62)
print("ML-09 VALIDATION AUDIT — AUTOMATED CHECKS PASSED")
print("PASS: Random-row and client-grouped runs use the same model/features.")
print("PASS: Client-grouped split has zero client overlap.")
print("PASS: Label-derived, ID, and outcome-window fields are excluded.")
print("PASS: Grouped predictions and error examples align with test labels.")
print("MANUAL: Verify predictor timestamps and review the current-run metrics.")
print("MANUAL: Run all cells from a clean kernel and commit the notebook.")
print("=" * 62)


ML-09 VALIDATION AUDIT — AUTOMATED CHECKS PASSED
PASS: Random-row and client-grouped runs use the same model/features.
PASS: Client-grouped split has zero client overlap.
PASS: Label-derived, ID, and outcome-window fields are excluded.
PASS: Grouped predictions and error examples align with test labels.
MANUAL: Verify predictor timestamps and review the current-run metrics.
MANUAL: Run all cells from a clean kernel and commit the notebook.


## Self-check before submission

- [x] Two paper findings and concrete, respectful methodology questions are included.
- [x] Random-row **before** and client-grouped **after** metrics are generated by this run.
- [x] Leakage audit includes a diagnostic-only `trend_pct` experiment and excludes it from the final model.
- [x] False-positive and false-negative examples are included without identifiers.
- [x] Claim rewrite uses measured, directional, decision-support language.
- [x] Runtime → Run all completes without errors.
- [x] Confirm predictor timestamps and the reconstructed historical-window assumptions.
- [x] Save/commit as `work/notebooks/w06_validation_audit.ipynb`, then submit your repository URL on the ML-09 card.

**Paper references:** finding #1 on page 6; finding #4 and the small-sample caveat on page 9. The repository data guidance notes that `trend_direction` is derived from `trend_pct`, IDs are not features, and client-grouped validation is preferred for repeated client data.
